# 01_standardize.ipynb
## Mục tiêu
Đây là giai đoạn đầu tiên trong quá trình processing dữ liệu thô. Những đoạn code sau sẽ làm những nhiệm vụ:
1. Parse các file json trong dataset.
2. Bổ xung thêm cột liên quan đến dự án.
3. Gộp mọi thứ lại trong một file csv để dễ đọc trong các quy trình sau.

## 1. Import library
Import các library như pandas và StringIO để đọc dữ liệu raw, đồng thời import requests để lấy thêm metadata từ các repo.

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime
import os
from io import StringIO
from concurrent.futures import ThreadPoolExecutor

## 2. Đọc dữ liệu
Load toàn bộ dataset vào memory sử dụng pandas.

In [2]:
DATASET_PATH = "../data/raw/comment_codeblock_dataset.jsonl"
DATASET_WITH_CHANGE_PATH = "../data/raw/comment_codeblock_first_change_dataset.jsonl"

full_dataset = pd.read_json(DATASET_PATH, lines=True)
dataset_with_change = pd.read_json(DATASET_WITH_CHANGE_PATH, lines=True)

dataset = full_dataset.merge(dataset_with_change, how="left")

print(dataset.head())
print(dataset_with_change.head())

                                            match_id  \
0  python__written_by_gpt__433861003__e397ad21a6e...   
1  python__written_by_gpt__433861003__e397ad21a6e...   
2  python__written_by_ai__437874902__14f89ee997ec...   
3  python__written_by_ai__437874902__14f89ee997ec...   
4  python__written_by_ai__437874902__1d766ed4953e...   

                                            trace_id          repo_full_name  \
0  python__written_by_gpt__433861003__e397ad21a6e...           flowerett/aoc   
1  python__written_by_gpt__433861003__e397ad21a6e...           flowerett/aoc   
2  python__written_by_ai__437874902__14f89ee997ec...  tipa16384/adventofcode   
3  python__written_by_ai__437874902__14f89ee997ec...  tipa16384/adventofcode   
4  python__written_by_ai__437874902__1d766ed4953e...  tipa16384/adventofcode   

                    path language  \
0  2022/python/d1-gpt.py   python   
1  2022/python/d1-gpt.py   python   
2        2017/puzzle2.py   python   
3        2017/puzzle2.py   python 

## 3. Merge datasets vào bảng dữ liệu
Define bảng dữ liệu output. Xử lý dataset gốc và chuyển nó thành bảng dữ liệu của riêng mình.

In [3]:
output = pd.DataFrame()

TOKEN = os.getenv("GITHUB_TOKEN")
if not TOKEN:
    raise RuntimeError(
        "Please set the GITHUB_TOKEN environment variable before running this cell."
    )

session = requests.Session()
session.headers.update(
    {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
    }
)

def fetch_graphql_batch(batch):
    query_parts = []
    for idx, (owner, repo, sha, file_path) in enumerate(batch):
        escaped_path = file_path.replace("\\", "\\\\").replace('"', '\\"')
        # Query file commit history up to 50 entries
        query_parts.append(
            f'c{idx}: repository(owner: "{owner}", name: "{repo}") {{ ' 
            f'  object(expression: "{sha}") {{ ' 
            f'    ... on Commit {{ ' 
            f'      history(path: "{escaped_path}", first: 50) {{ ' 
            f'        nodes {{ oid committedDate }} ' 
            f'      }} ' 
            f'    }} ' 
            f'  }} ' 
            f'}}'
        )

    query = f"query {{ {' '.join(query_parts)} }}"

    try:
        r = session.post(
            "https://api.github.com/graphql", json={"query": query}, timeout=30
        )
        if r.status_code != 200:
            return [(None, None)] * len(batch)

        data = r.json().get("data", {}) or {}
        results = []
        for idx in range(len(batch)):
            repo_data = data.get(f"c{idx}") or {}
            obj = repo_data.get("object") or {}
            history = obj.get("history") or {}
            nodes = history.get("nodes") or []
            if nodes:
                # nodes are reverse-chronological (newest to oldest).
                # The last entry nodes[-1] is the earliest commit where this file was introduced.
                introductory_commit = nodes[-1]
                oid = introductory_commit.get("oid")
                date = introductory_commit.get("committedDate")
                formatted_date = (
                    pd.to_datetime(date, utc=True).strftime("%Y-%m-%dT%H:%M:%SZ")
                    if date
                    else None
                )
                results.append((oid, formatted_date))
            else:
                results.append((None, None))
        return results
    except requests.RequestException:
        return [(None, None)] * len(batch)

def get_all_commit_info(file_urls, file_paths, batch_size=25):
    parsed_all = []
    for url, path in zip(file_urls, file_paths):
        if pd.isna(url) or pd.isna(path):
            parsed_all.append((None, None, None, None))
            continue
        parts = str(url).split("/")
        if len(parts) >= 7:
            parsed_all.append((parts[3], parts[4], parts[6], str(path)))
        else:
            parsed_all.append((None, None, None, None))

    unique = [item for item in dict.fromkeys(parsed_all) if item[0] is not None]
    batches = [unique[i : i + batch_size] for i in range(0, len(unique), batch_size)]

    info_map = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        for batch, batch_results in zip(batches, executor.map(fetch_graphql_batch, batches)):
            for key, (oid, date) in zip(batch, batch_results):
                info_map[key] = (oid, date)

    commit_urls = []
    commit_dates = []
    for item in parsed_all:
        if item[0] is None:
            commit_urls.append(None)
            commit_dates.append(None)
            continue
        owner, repo, _, _ = item
        real_oid, real_date = info_map.get(item, (None, None))
        if real_oid:
            commit_urls.append(f"https://github.com/{owner}/{repo}/commit/{real_oid}")
        else:
            commit_urls.append(None)
        commit_dates.append(real_date)

    return commit_urls, commit_dates

real_commit_urls, real_commit_dates = get_all_commit_info(dataset["file_url"], dataset["path"])

output["id"] = dataset["match_id"]
output["repo"] = dataset["repo_full_name"]
output["language"] = dataset["language"]
output["commit_url"] = real_commit_urls
output["file"] = dataset["path"]
output["code_block"] = dataset["code_block"].apply(lambda block: block.replace("\n", "\\n"))
output["commit_date"] = real_commit_dates
output["change_commit_url"] = dataset["first_change_commit_url"]
output["change_commit_date"] = dataset["first_change_date"]

print(output.head())

                                                  id                    repo  \
0  python__written_by_gpt__433861003__e397ad21a6e...           flowerett/aoc   
1  python__written_by_gpt__433861003__e397ad21a6e...           flowerett/aoc   
2  python__written_by_ai__437874902__14f89ee997ec...  tipa16384/adventofcode   
3  python__written_by_ai__437874902__14f89ee997ec...  tipa16384/adventofcode   
4  python__written_by_ai__437874902__1d766ed4953e...  tipa16384/adventofcode   

  language                                         commit_url  \
0   python  https://github.com/flowerett/aoc/commit/52a0fb...   
1   python  https://github.com/flowerett/aoc/commit/52a0fb...   
2   python  https://github.com/tipa16384/adventofcode/comm...   
3   python  https://github.com/tipa16384/adventofcode/comm...   
4   python  https://github.com/tipa16384/adventofcode/comm...   

                    file                                         code_block  \
0  2022/python/d1-gpt.py  # solution written by G

## 4. Export bảng dữ liệu
Export vào folder interim trong data để tiếp tục clean.

In [4]:
INTERIM_DIR = "../data/interim"
if not os.path.exists(INTERIM_DIR):
    os.makedirs(INTERIM_DIR)
    
PROCESSED_DATA_PATH = f"{INTERIM_DIR}/standardized_data_{datetime.today().strftime('%Y-%m-%d %H_%M_%S')}.csv"
output.to_csv(PROCESSED_DATA_PATH, index=False)
output.to_csv(f"{INTERIM_DIR}/latest_standardized_data.csv", index=False)